In [ ]:
import requests
import datetime 
from datetime import date
from datetime import timedelta
import time
import pandas as pd




In [ ]:
ruta_base = "../data/raw/omie"
fecha_inicial = date(2023, 1, 1)
fecha_final = date(2026, 6, 30)
fecha_actual = fecha_inicial
lista_fechas = []
while fecha_actual <= fecha_final:
    fecha_formateada = fecha_actual.strftime("%Y%m%d")
    lista_fechas.append(fecha_formateada)
    fecha_actual = fecha_actual + timedelta(days=1)

print(len(lista_fechas))
print(lista_fechas[:3])
print(lista_fechas[-3:])
    
for fecha in lista_fechas:
     url = f"https://www.omie.es/es/file-download?parents=marginalpdbc&filename=marginalpdbc_{fecha}.1"
     r = requests.get(url)
     if r.status_code == 200:
        with open(f"{ruta_base}/marginalpdbc_{fecha}.1", "wb") as f:
         f.write(r.content)
        
     else: 
        print(f"Error al descargar el archivo para la fecha {fecha}. Código de estado: {r.status_code}")
     time.sleep(.5)    




In [ ]:
lista_tablas = []
names=["ano", "mes", "dia", "hora", "precio_espana","precio_portugal","vacia"]
for fecha in lista_fechas:
    try:
        datos_omie = pd.read_csv(
            f"{ruta_base}/marginalpdbc_{fecha}.1",
            sep=";", 
            skiprows=1, 
            skipfooter=1, 
            names=names, 
            engine="python"
        )
        lista_tablas.append(datos_omie)
    except FileNotFoundError:
        print(f"No se encontró fichero para la fecha {fecha}")
    
tabla_omie = pd.concat(lista_tablas, ignore_index=True)
print(tabla_omie.shape)
tabla_omie.head()
tabla_omie.tail()


## Nota: cambio de resolución horaria a cuartohoraria (15 minutos)

El 30 de septiembre de 2025, el mercado eléctrico diario de la Unión Europea pasó de operar por horas a operar cada 15 minutos, como parte del Acoplamiento Único Diario (SDAC). Esto significa que los precios ya no se calculan 24 veces al día (una por hora), sino 96 veces al día, en intervalos de 15 minutos, alineando el mercado español con el resto de Europa.

**Origen regulatorio:** el cambio se enmarca en el Reglamento europeo 2017/2195 y se implementó en España a partir de las reglas publicadas por la CNMC (Resolución de 28 de febrero de 2025), que adaptan el mercado diario e intradiario español a la "negociación cuarto-horaria" (MTU15).

**Fuente oficial:** Resolución de 28 de febrero de 2025 de la CNMC, por la que se publican las reglas de funcionamiento de los mercados diario e intradiario de electricidad para su adaptación a la negociación cuarto-horaria — [BOE-A-2025-4908](https://www.boe.es/diario_boe/txt.php?id=BOE-A-2025-4908)

**Motivación del cambio:** reflejar con mayor precisión la generación y demanda de electricidad previstas en el sistema, haciendo el sistema eléctrico europeo más flexible y mejor preparado para la creciente cuota de energía renovable. 

**Impacto en este dataset:** los ficheros de OMIE anteriores al 30/09/2025 tienen 24 filas por día (columna `hora` de 1 a 24), mientras que los ficheros posteriores a esa fecha tienen 96 filas por día (columna `hora`/`periodo` de 1 a 96, representando intervalos de 15 minutos). Esto se tendrá en cuenta en la fase de ETL para homogeneizar ambos formatos en un único timestamp real.


In [ ]:
with open(f"{ruta_base}/marginalpdbc_20230615.1", "r") as f:
    contenido = f.read()
print(contenido)

In [ ]:
tabla_omie.head()
tabla_omie.tail()

In [ ]:
anos=tabla_omie["ano"]
anos.unique()

In [ ]:
tabla_omie.to_csv(r"C:\Users\jaime\Desktop\Jaime_LLorca\proyectos\prediccion-electrica\data\interim\tabla_OMIE", index=False, encoding="utf-8")